# ニューラルネットワークモデル

## 今回の流れ

最近のLLMは手法の解説やコードの高い説明能力を持ち、
各人が、自分で大規模言語モデル（LLM）を用いて、ニューラルネットワークモデル手法を学習、コード生成ができます。
**この節ではそれを行ってみます。**


本資料では、コードの詳細な説明は行いません。必要に応じて、各自がLLMに質問し、詳細な解説を得てください。

数式や文章だけでは理解しにくい箇所について、私が図を用いて補足説明します。


今回は、簡単なニューラルネットワークモデル,autoencoder, variational autoencoder, GAN, diffusion modelのLLMによる説明を私がまとめています。


今回の話の流れは以下になります。
まず、簡単なニューラルネットワークモデル

次に、Autoencoder
そして、Variational Autoencoder
最後にdiffusion modelを紹介します。
GANは生成モデル例として途中で紹介しています。


![](fig/8200-intro.jpg)

最後のDiffusion modelは画像生成AIとして聞いたことがあると思います。


## PyTorchのインストール
本教材を実行するにはPyTorchのインストールが必要です。

Condaによるインストールは勧めません。動作するPythonのversionが限られます。
Pipを用いてインストールすることを勧めます。

インストールの仕方については公式ホームページhttps://pytorch.org/get-started/locally/
の上の方に以下の選択画面があります。

![](fig/8200-Pytorch-install.jpg)

1. 赤い部分を各自選択する。

2. 一番下の”Run this Command”にpip コマンドが表示される。

しかし、例えば、CPUの場合は
$ pip torch
で良いはずです。GPU (CUDA)を用いる場合はCompute　PlatformでCUDAのあなたのPCのversionを選択してください。



### 大規模言語モデル（LLM）を用いたニューラルネットワークモデルの学習


ニューラルネットワークを使った手法は数が多く、把握するのが大変です。

幸いなことに、有名なニューラルネットワークモデルは世の中に初心者向けのコード例、解説例が多いためLLMが妥当な回答を行うことができます。

今回はLLMを用いてニューラルネットワークの定義の仕方と、PyTorchでのモデル学習（訓練）の仕方を紹介します。
ここではChatGPTをLLMの例として用います。




## 復習


回帰モデル、分類モデルにかかわらず、
何かの最適化関数𝐿があり、𝐿を最小化するようにモデルを学習する。
𝐿に正則化項、拘束条件を加えることもある。


ニューラルネットワークモデルの場合𝐿は損失関数と呼ばれる。


## 説明順序の違い

教科書は、後ろを参照しないように、
細部→細部をまとめた手法→目的、という順で説明を行います。

一方、ChatGPTの説明はそれとは逆になります。
目的←そのための手法←手法の細部の説明の順です。
そして不明事項（細部）を更に尋ねていくことになります。


## 説明形式
ChatGPTへの質問→回答という順に説明を行います。
講演者自体この形式は面倒で、回答のみをまとめた方が楽なのですが、各自が「 ChatGPTへの質問→回答」という順で知識獲得を行う例とするためこの形式で説明を行います。

受講者が将来、他の手法に対して、ChatGPTを用いて同様に知識獲得できることを期待します。


以下はgpt-4oの回答です。

### prompt
```
# 私について
私は機械学習手法の初心者です。
# 依頼
ニューラルネットワークモデルの学習過程を初心者に分かるように教えて下さい。数式を用いてください。
```

### answer

##### ニューラルネットワークの基本構造

ニューラルネットワークは、以下のふたつの要素から成り立っています。

    ニューロン（ノード）: 各層でデータを処理する基本単位。
    層: ニューロンが階層的に配置されたもの。一般に、入力層、隠れ層、出力層の3つの層があります。

具体的には、次のようになります：

    入力層: 入力データが最初に入る場所。
    隠れ層: 入力を処理し、出力に変換する中間層。
    出力層: 最終結果を出す層。分類結果や予測値を出力します。

![](fig/8200-NN-intro.jpg)

##### 学習過程

ニューラルネットワークは以下の4つのステップで学習を行います。

1. 前向き伝播（Forward Propagation）:
2. 損失の計算（Loss Calculation）:
3. 逆伝播（Backpropagation）:
4. 重複（Epochs):

#### 1. 前向き伝播（Forward Propagation）

* 入力データを受け取り、隠れ層および出力層を通して出力を計算します。

**数式：**

$$
z^{(l)} = W^{(l)} a^{(l-1)} + b^{(l)}
$$

$$
a^{(l)} = \sigma\left(z^{(l)}\right)
$$

ここで、

- $z^{(l)}$ は層 $l$ のニューロンの入力。
- $W^{(l)}$ は重み行列。
- $a^{(l)}$ は層 $l$ の出力（活性化）。
- $b^{(l)}$ はバイアス。
- $\sigma(z)$ は活性化関数（例：シグモイド、ReLU）。

![](fig/8200-NN-explain1.jpg)

#### 2. 損失の計算（Loss Calculation）

* 出力と実際の正解ラベルとの間の誤差を計算します。

数式：

$$
L(y, \hat{y}) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y^{(i)} \log\left(\hat{y}^{(i)}\right) + \left(1 - y^{(i)}\right) \log\left(1 - \hat{y}^{(i)}\right) \right]
$$

ここで、

* $L$ は損失関数。
* $y^{(i)}$ は実際のラベル。
* $\hat{y}^{(i)}$ は予測された出力。

***(補足: これは主としてlabel(分類）用の損失関数です。後で再び説明が出てきます。）***


#### 3. 逆伝播（Backpropagation）

* 損失を最小化するために、ネットワークの重みやバイアスを調整します。これは勾配降下法を使用して行います。

**数式：**

$$
W^{(l)} \leftarrow W^{(l)} - \eta \frac{\partial L}{\partial W^{(l)}}
$$

$$
b^{(l)} \leftarrow b^{(l)} - \eta \frac{\partial L}{\partial b^{(l)}}
$$

ここで、

* $W^{(l)}$ は層 $l$ の重み行列。
* $b^{(l)}$ は層 $l$ のバイアス。
* $\eta$ は学習率。
* $\dfrac{\partial L}{\partial W^{(l)}}$ と $\dfrac{\partial L}{\partial b^{(l)}}$ は損失関数 $L$ の勾配。


![](fig/8200-NN-backpropagation.jpg)


#### 4. 重複（Epochs）

* 上記のステップを何度も繰り返します。各繰り返しをエポックと呼びます。


### prompt
```prompt
回帰モデルと分類モデルの損失関数の例を教えてください。
```


#### 注意
以下では予測値にhatをつけています。

### answer

回帰モデルの損失関数

回帰モデルは連続値を予測するため、主に以下の損失関数が使われます。




---

1. 平均二乗誤差（Mean Squared Error, MSE）

* 予測と実際の値との差を二乗し、平均を取ります。この関数は、予測誤差が大きいほど損失が大きくなり、外れ値に敏感です。

**数式：**

$$
\mathrm{MSE} = \frac{1}{N} \sum_{i=1}^{N} \left( y^{(i)} - \hat{y}^{(i)} \right)^2
$$

ここで、

* $N$ はデータポイントの数。
* $y^{(i)}$ は実際の値。
* $\hat{y}^{(i)}$ は予測された値。



### 分類モデルの損失関数

分類モデルはカテゴリを予測するため、以下の損失関数が一般的に使われます。

---

1. **バイナリクロスエントロピー（Binary Cross-Entropy）**

* 2クラス分類問題に使用され、実際のラベルと予測された確率の間の誤差を計算します。

**数式：**

$$
\mathrm{Loss}
= -\frac{1}{N} \sum_{i=1}^{N}
\left[ y^{(i)} \log\left(\hat{y}^{(i)}\right) + 
\left(1 - y^{(i)}\right)\log\left(1 - \hat{y}^{(i)}\right) \right]
$$

---

2. **カテゴリカルクロスエントロピー（Categorical Cross-Entropy）**

* 多クラス分類問題に使用され、実際のカテゴリの確率分布と予測された確率分布との間の誤差を計算します。

**数式：**

$$
\mathrm{Loss}
= -\sum_{c=1}^{C} y_c \log\left(\hat{y}_c\right)
$$

ここで、

* $C$ はクラスの数。
* $y_c$ は実際のクラスの one-hot エンコーディングの値。
* $\hat{y}_c$ は予測されたクラスの確率。





「one-hot エンコーディング」という言葉が出てきます。
これは何でしょう？

#### prompt
```prompt
one-hot エンコーディングを具体的に教えて下さい。
```

#### answer
```answer
クラスの数がCの場合、期待されるラベルはC次元のベクトルで、正しいクラスのインデックスだけが1で、他は0となります。
```
と答えます。さらに、

#### prompt
```prompt
one-hot エンコーディングを具体的に教えて下さい。
```


#### answer
と質問すると、例えば、クラスの数C=3で、（犬、猫、鳥）として、
ラベルが犬の場合は$𝑦=(1, 0, 0)$→$𝑦_1  = 1, 𝑦_2  = 0, 𝑦_3  = 0$
ラベルが猫の場合は$𝑦=(0, 1, 0)$→$𝑦_1  =0, 𝑦_2  =1, 𝑦_3  = 0$
とする、と回答が得られます。𝑦 ̂も上の表示に倣います。

２クラスの場合がバイナリクロスエントロピーと等価であることが理解できるでしょう。
